#### Near-OOD-Auswertung -- HuffPost (Lifestyle-Kategorien) als Out-of-Distribution-Referenz

Analog zu `07_ood_evaluation.ipynb` (IMDB als Far-OOD), aber mit einer methodisch anderen Frage: Statt eines komplett fremden Genres (Filmkritiken) verwenden wir hier **echte Nachrichtenschlagzeilen** (gleiches journalistisches Format wie AG News) aus vier Themenkategorien, die es in AG News NICHT gibt (`TRAVEL`, `STYLE & BEAUTY`, `PARENTING`, `FOOD & DRINK`, aus dem HuffPost News Category Dataset). Das isoliert den Themen-Effekt vom Genre-Effekt -- ein haerterer, methodisch saubererer OOD-Test als IMDB.

Laedt dieselben bereits gespeicherten Posterior-Samples wie `07_ood_evaluation.ipynb` (kein neues Fitting) und wertet sie zusaetzlich auf den HuffPost-Texten aus. Wie bei IMDB: nur die label-unabhaengigen Unsicherheits-Kennzahlen (Predictive Entropy, Expected Entropy, Mutual Information) sind fuer OOD gueltig, keine Accuracy/NLL/Brier.

**Update:** MILE zeigt jetzt auf den REPARIERTEN Multi-Chain-Lauf bei der echten N_PER_CLASS=2048-Skala (Chain 1 per Parameter-Swap repariert, Donor = Mittelwert aus Chain 2+3 -- siehe `14_chain1_paramswap_test_n2048.ipynb` und `15_n2048_repaired_k3_full_eval.ipynb`). Der fruehere, unreparierte Lauf zeigte hier ein stark invertiertes AUROC(MI) (0.0021) -- dasselbe Chain-Tuning-Instabilitaetsproblem wie bei der urspruenglichen Chain-2-Diagnose, nur diesmal Chain 1 betreffend. Mit dem reparierten Lauf sollte das jetzt ein sinnvolles, nicht-invertiertes AUROC(MI) liefern.


In [1]:
from pathlib import Path

# ============================================================
# CONFIG
# NEAR-OOD EVALUATION (HuffPost -- TRAVEL, STYLE & BEAUTY, PARENTING, FOOD & DRINK)
# ============================================================

MASTER_DIR = Path(
    "/dss/dsshome1/00/ra58vit2/Masterarbeit"
)

QWEN_REPO = MASTER_DIR / "bayes_sub_inf"

BASELINE_SCRIPT = (
    QWEN_REPO / "experiments" / "ag_news_qwen_lora" / "evaluate_saved_baseline.py"
)

MODEL_PATH = (
    QWEN_REPO / "tests" / "qwen_data" / "helper_scripts" / "data_store"
    / "models" / "qwen2.5_0.5B_float16"
)

RESULTS_ROOT = MASTER_DIR / "method_results"
OOD_RESULT_PATH = RESULTS_ROOT / "ood_huffpost_near_comparison.json"

# WICHTIG: eigenes Cache-Verzeichnis, getrennt von 07_ood_evaluation.ipynb (IMDB)!
# Die run_names sind identisch zum IMDB-Notebook (gleiche Methoden/Posterior-Samples),
# nur die OOD-Quelle unterscheidet sich -- bei gemeinsamem Cache-Ordner wuerden sich
# die pro-run_name JSON-Zwischenspeicherungen sonst gegenseitig ueberschreiben bzw.
# faelschlich den IMDB-Wert fuer HuffPost zurueckgeben.
OOD_CACHE_DIR = RESULTS_ROOT / "ood_cache_near_huffpost"

# ------------------------------------------------------------
# OOD-Daten (HuffPost) -- gleiche Tokenisierung wie das AG-News-Testset:
# derselbe Qwen-Tokenizer, gleiche Sequenzlaenge, gleiches Padding.
# SEQ_LEN=128, identisch zu 07_ood_evaluation.ipynb (IMDB), damit das
# Format zum AG-News-Testset passt und beide OOD-Sets direkt vergleichbar sind.
# ------------------------------------------------------------

SEQ_LEN = 128

# Vier Kategorien, die es in AG News (World/Sports/Business/Sci-Tech) nicht gibt --
# bewusst als Near-OOD gewaehlt: gleiches journalistisches Format wie AG News,
# nur thematisch neu. Kategorie-Strings muessen exakt zum HuffPost-Datensatz passen
# (Grossschreibung inkl. "&") -- wird in der Lade-Zelle unten validiert.
HUFFPOST_CATEGORIES = ["TRAVEL", "STYLE & BEAUTY", "PARENTING", "FOOD & DRINK"]
N_PER_CATEGORY = 500          # 4 x 500 = 2000, gleich gross wie das IMDB-OOD-Set
N_OOD_EXAMPLES = len(HUFFPOST_CATEGORIES) * N_PER_CATEGORY
HUFFPOST_SAMPLE_SEED = 42

EVAL_BATCH_SIZE = 64

# ------------------------------------------------------------
# Welche gespeicherten Runs ausgewertet werden -- IDENTISCH zu
# 07_ood_evaluation.ipynb (IMDB), damit beide OOD-Sets exakt dieselben
# Methoden/Posterior-Samples verwenden und direkt vergleichbar sind.
#
# MILE zeigt jetzt auf den REPARIERTEN Multi-Chain-Lauf bei der ECHTEN
# N_PER_CLASS=2048-Skala (Chain 1 per Parameter-Swap repariert, Donor =
# Mittelwert aus Chain 2+3 -- siehe 14_chain1_paramswap_test_n2048.ipynb
# und 15_n2048_repaired_k3_full_eval.ipynb). Alter Eintrag ersetzt, nicht
# nur ergaenzt (war falsch beschriftet UND unrepariert).
#
# MFVI und Laplace zeigen jetzt ebenfalls auf die ECHTEN N_PER_CLASS=2048-
# Laeufe ("..._seed1_balanced8192_..."). Die alten Eintraege ("..._balanced2048_...",
# ohne seed1-Praefix) entsprechen wegen derselben Namenskonvention
# (balanced{4*N_PER_CLASS}) tatsaechlich N_PER_CLASS=512, nicht 2048 --
# trotz der irrefuehrenden "(2048)"-Beschriftung im alten Label.
# ------------------------------------------------------------

RUNS = [
    ("MILE (Multi-Chain, K=3, N=2048, repariert)", RESULTS_ROOT / "mile_qwen_agnews",
     "mile_multichain_FIXED_chain1paramswap_donor23mean_balanced8192_seq32_prior0.02_warmup250_spc30_thin30_testeval"),
    ("MFVI (N=2048)", RESULTS_ROOT / "mfvi_qwen_agnews",
     "mfvi_seed1_balanced8192_seq32_prior0.02_steps200_samples10_testeval"),
    ("Laplace (N=2048)", RESULTS_ROOT / "laplace_qwen_agnews",
     "laplace_seed1_balanced8192_seq32_prior0.02_fscale1.0_samples10"),
    ("Deep Ensemble (N=3)", RESULTS_ROOT / "deep_ensemble_qwen_agnews",
     "deep_ensemble_n3_seq32"),
    ("pSMILE (Mini-Batch, K=3, N=2048)", RESULTS_ROOT / "mile_qwen_agnews",
     "mile_psmile_k3_prior0.02_mb32_spc90_thin30_testeval"),
]

print("Konfigurierte Runs:", len(RUNS))
for label, result_dir, run_name in RUNS:
    print(" -", label, "->", run_name)
print("HuffPost-Kategorien:", HUFFPOST_CATEGORIES)
print("N pro Kategorie:", N_PER_CATEGORY, " -> N_OOD_EXAMPLES:", N_OOD_EXAMPLES)


Konfigurierte Runs: 5
 - MILE (Multi-Chain, K=3, N=2048, repariert) -> mile_multichain_FIXED_chain1paramswap_donor23mean_balanced8192_seq32_prior0.02_warmup250_spc30_thin30_testeval
 - MFVI (N=2048) -> mfvi_seed1_balanced8192_seq32_prior0.02_steps200_samples10_testeval
 - Laplace (N=2048) -> laplace_seed1_balanced8192_seq32_prior0.02_fscale1.0_samples10
 - Deep Ensemble (N=3) -> deep_ensemble_n3_seq32
 - pSMILE (Mini-Batch, K=3, N=2048) -> mile_psmile_k3_prior0.02_mb32_spc90_thin30_testeval
HuffPost-Kategorien: ['TRAVEL', 'STYLE & BEAUTY', 'PARENTING', 'FOOD & DRINK']
N pro Kategorie: 500  -> N_OOD_EXAMPLES: 2000


In [2]:
# ============================================================
# IMPORTS AND ENVIRONMENT (selbstheilend -- installiert datasets/
# transformers per `uv add` nach, falls sie fehlen sollten)
# ============================================================

import os
import sys
import json
import runpy
import subprocess

import numpy as np
import jax
import jax.numpy as jnp
from jax import random

os.chdir(QWEN_REPO)

if str(QWEN_REPO) not in sys.path:
    sys.path.insert(0, str(QWEN_REPO))

print("Python:", sys.executable)
print("JAX:", jax.__version__)
print("Devices:", jax.devices())

assert QWEN_REPO.exists()
assert BASELINE_SCRIPT.exists()
assert MODEL_PATH.exists(), f"Modellpfad nicht gefunden: {MODEL_PATH}"

UV_PATH = os.path.expanduser("~/.local/bin/uv")


def uv_add(package_name):
    print(f"uv add {package_name} (in {QWEN_REPO}) ...")
    result = subprocess.run(
        [UV_PATH, "add", package_name],
        cwd=str(QWEN_REPO),
        capture_output=True, text=True,
    )
    print(result.stdout[-1500:])
    if result.returncode != 0:
        print(result.stderr[-1500:])
        raise RuntimeError(f"uv add {package_name} fehlgeschlagen.")
    print(f"'{package_name}' hinzugefuegt.")


for _ in range(3):
    try:
        import datasets
        import transformers
        break
    except ModuleNotFoundError as exc:
        print(f"Fehlendes Modul: {exc.name}")
        uv_add(exc.name)

print("datasets:", datasets.__version__)
print("transformers:", transformers.__version__)


Python: /dss/dsshome1/00/ra58vit2/Masterarbeit/bayes_sub_inf/.venv/bin/python3
JAX: 0.5.3


Devices: [CudaDevice(id=0)]


datasets: 2.21.0
transformers: 4.44.2


In [3]:
# ============================================================
# GETEILTES MODUL
# ============================================================

import qwen_posterior_utils as qpu

qpu.patch_subspace_curve_predict()


Patched SubspaceBaseModel._predict: lax.cond bypass for concrete train flags active.


In [4]:
# ============================================================
# LOAD QWEN AG-NEWS BASELINE (env/data + LoRA-Struktur)
# ============================================================

baseline_objects = runpy.run_path(str(BASELINE_SCRIPT))

env = baseline_objects["env"]
params = baseline_objects["params"]
data = baseline_objects["data"]
rng_key = baseline_objects["rng_key"]

print("Baseline geladen. Model:", type(env.s_model))


Dataset loaded: 120000 samples, 4 classes
Target token IDs: [[15]
 [16]
 [17]
 [18]]


Validation data are used
Training data: 108000 samples, Validation data: 12000 samples
Computed cp_fix: [1]
Setting num_epochs to 1 to match 500 steps


Using cosine rho schedule with total steps 500
save best parameters
Loaded checkpoint: tmp_files/955nvvay_pretrained_params.npy
Saved object shape: (1,)


post_predict_logits shape: (12000, 4)


post_predict_logits shape: (12000, 4)


post_predict_logits shape: (7600, 4)


post_predict_logits shape: (7600, 4)

SAVED BEST CHECKPOINT - VALIDATION
val_ll: -0.246199
val_acc: 0.913667
val_ece: 0.009229
val_brier: 0.131579
val_mutual_information: 0.000000
val_mean_entropy: 0.000000

SAVED BEST CHECKPOINT - TEST
test_ll: -0.251077
test_acc: 0.914737
test_ece: 0.007497
test_brier: 0.131565
test_mutual_information: 0.000000
test_mean_entropy: 0.000000
Baseline geladen. Model: <class 'subspace_inference.curve_optimizer.subspace_curve.LoRACategorySubspace'>


In [5]:
# ============================================================
# REBUILD_FULL_PARAMS (nur die LoRA-Struktur -- identisch fuer
# alle Methoden, siehe 06_evaluate_all.ipynb)
# ============================================================

theta_map, all_leaves, lora_indices, full_tree_definition, unravel_lora = (
    qpu.vectorize_lora_params(params["params"])
)

rebuild_full_params = qpu.build_rebuild_fn(
    all_leaves, lora_indices, full_tree_definition, unravel_lora,
)

print("theta_map shape:", theta_map.shape)


LoRA objects: 48
Sampling vector shape: (540672,)
Sampling parameters: 540672
All MAP parameters finite: True
theta_map shape: (540672,)


In [6]:
# ============================================================
# ID-DATEN (AG News Testset) -- Referenzformat + spaeter ID-Seite
# des ID-vs-OOD-Vergleichs
# ============================================================

x_id, y_id = data.get("test")

print("ID (AG News Test) Beispiele:", int(y_id.shape[0]))
print("ID input_ids shape:", x_id["input_ids"].shape, x_id["input_ids"].dtype)
print("ID attention_mask shape:", x_id["attention_mask"].shape, x_id["attention_mask"].dtype)


ID (AG News Test) Beispiele: 7600
ID input_ids shape: (7600, 128) int32
ID attention_mask shape: (7600, 128) int32


In [7]:
# ============================================================
# OOD-DATEN (HuffPost, 4 Lifestyle-Kategorien) LADEN + TOKENISIEREN
# (gleicher Tokenizer, gleiche SEQ_LEN/Padding wie das AG-News-Testset)
# ============================================================

from datasets import load_dataset
from transformers import AutoTokenizer

# Tokenizer direkt von HuggingFace -- identisch zu 07_ood_evaluation.ipynb.
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token or tokenizer.bos_token

# ACHTUNG: der genaue HF-Dataset-Pfad fuer das HuffPost News Category
# Dataset ist nicht zu 100% sicher -- falls der erste Pfad hier nicht
# funktioniert (Datensatz umbenannt/entfernt), auf dem Cluster direkt
# auf https://huggingface.co/datasets?search=news-category nach einem
# aktuellen Mirror suchen und HUFFPOST_HF_PATH unten anpassen.
HUFFPOST_HF_PATH = "heegyu/news-category-dataset"

try:
    huffpost = load_dataset(HUFFPOST_HF_PATH)
except Exception as exc:
    raise RuntimeError(
        f"Konnte '{HUFFPOST_HF_PATH}' nicht laden ({exc}). "
        "Pruefe auf dem Cluster den aktuellen HF-Dataset-Pfad fuer das "
        "HuffPost News Category Dataset und passe HUFFPOST_HF_PATH an."
    )

# Alle Splits zusammenfassen, falls das Dataset in mehrere Splits aufgeteilt ist.
if isinstance(huffpost, dict) or hasattr(huffpost, "keys"):
    split_names = list(huffpost.keys())
    print("Gefundene Splits:", split_names)
    huffpost_all = datasets.concatenate_datasets([huffpost[s] for s in split_names])
else:
    huffpost_all = huffpost

available_categories = sorted(set(huffpost_all["category"]))
print(f"Kategorien im Datensatz gefunden: {len(available_categories)}")

missing = [c for c in HUFFPOST_CATEGORIES if c not in available_categories]
if missing:
    raise ValueError(
        f"Diese Kategorien fehlen im geladenen Datensatz: {missing}. "
        f"Verfuegbare Kategorien (Auszug): {available_categories[:20]} ... "
        "Kategorie-Strings in HUFFPOST_CATEGORIES anpassen (Gross-/Kleinschreibung!)."
    )

huffpost_texts = []
for category in HUFFPOST_CATEGORIES:
    subset = huffpost_all.filter(lambda row: row["category"] == category)
    subset = subset.shuffle(seed=HUFFPOST_SAMPLE_SEED)
    n_available = len(subset)
    if n_available < N_PER_CATEGORY:
        raise ValueError(
            f"Kategorie '{category}' hat nur {n_available} Beispiele, "
            f"gebraucht werden {N_PER_CATEGORY}."
        )
    subset = subset.select(range(N_PER_CATEGORY))
    for headline, description in zip(subset["headline"], subset["short_description"]):
        headline = (headline or "").strip()
        description = (description or "").strip()
        text = f"{headline} {description}".strip() if description else headline
        huffpost_texts.append(text)
    print(f"  {category}: {N_PER_CATEGORY} Beispiele gezogen (von {n_available} verfuegbar)")

print("HuffPost Beispiele geladen (gesamt):", len(huffpost_texts))
assert len(huffpost_texts) == N_OOD_EXAMPLES

encoded = tokenizer(
    huffpost_texts,
    padding="max_length",
    truncation=True,
    max_length=SEQ_LEN,
    return_tensors="np",
)

x_ood = {
    "input_ids": jnp.asarray(encoded["input_ids"]),
    "attention_mask": jnp.asarray(encoded["attention_mask"]),
}

# Dummy-Labels -- werden NIE fuer Accuracy/NLL/Brier verwendet (HuffPost hat
# keine gueltigen AG-News-Klassen), nur Platzhalter fuer die gemeinsame
# Funktionssignatur.
y_ood = jnp.zeros((len(huffpost_texts),), dtype=jnp.int32)

print("OOD input_ids shape:", x_ood["input_ids"].shape, x_ood["input_ids"].dtype)
print("OOD attention_mask shape:", x_ood["attention_mask"].shape, x_ood["attention_mask"].dtype)


Gefundene Splits: ['train']
Kategorien im Datensatz gefunden: 42


  TRAVEL: 500 Beispiele gezogen (von 9900 verfuegbar)
  STYLE & BEAUTY: 500 Beispiele gezogen (von 9814 verfuegbar)
  PARENTING: 500 Beispiele gezogen (von 8791 verfuegbar)
  FOOD & DRINK: 500 Beispiele gezogen (von 6340 verfuegbar)
HuffPost Beispiele geladen (gesamt): 2000


OOD input_ids shape: (2000, 128) int32
OOD attention_mask shape: (2000, 128) int32


In [8]:
# ============================================================
# FORMAT-VALIDIERUNG -- x_ood muss strukturell zu x_id passen,
# sonst schlaegt die Auswertung weiter unten mysterioes fehl.
# ============================================================

assert set(x_ood.keys()) == set(x_id.keys()), (
    f"Key-Mismatch: x_ood hat {set(x_ood.keys())}, x_id hat {set(x_id.keys())}"
)
assert x_ood["input_ids"].shape[1] == x_id["input_ids"].shape[1], (
    f"Sequenzlaenge passt nicht: OOD={x_ood['input_ids'].shape[1]}, "
    f"ID={x_id['input_ids'].shape[1]}"
)

print("Format-Check OK -- x_ood ist strukturell kompatibel mit x_id.")
print("ID examples: ", int(y_id.shape[0]))
print("OOD examples:", int(y_ood.shape[0]))


Format-Check OK -- x_ood ist strukturell kompatibel mit x_id.
ID examples:  7600
OOD examples: 2000


In [9]:
# ============================================================
# AUROC (Mann-Whitney-U-Formel, keine sklearn-Abhaengigkeit)
# Hoeherer Score = wird als "eher OOD" interpretiert.
# ============================================================

def compute_auroc(id_scores, ood_scores):
    id_scores = np.asarray(id_scores, dtype=np.float64)
    ood_scores = np.asarray(ood_scores, dtype=np.float64)

    all_scores = np.concatenate([id_scores, ood_scores])
    labels = np.concatenate([
        np.zeros(len(id_scores)), np.ones(len(ood_scores)),
    ])

    order = np.argsort(all_scores)
    ranks = np.empty(len(all_scores), dtype=np.float64)
    ranks[order] = np.arange(1, len(all_scores) + 1)

    n_pos = len(ood_scores)
    n_neg = len(id_scores)
    sum_ranks_pos = ranks[labels == 1].sum()
    u_stat = sum_ranks_pos - n_pos * (n_pos + 1) / 2

    return float(u_stat / (n_pos * n_neg))


In [10]:
# ============================================================
# EVALUATE ALL RUNS -- ID (AG News Test) UND OOD (HuffPost, Near-OOD).
#
# ID-Wahrscheinlichkeiten werden, falls vorhanden, aus der bereits
# gespeicherten {run_name}_probabilities.npy geladen statt neu berechnet.
#
# OOD-Ergebnisse werden pro Methode in OOD_CACHE_DIR zwischengespeichert
# (eigener Ordner, getrennt vom IMDB-Cache -- siehe Config-Zelle).
# ============================================================

results = []

rng_key, eval_master_key = random.split(rng_key)

OOD_CACHE_DIR.mkdir(parents=True, exist_ok=True)

for label, result_dir, run_name in RUNS:
    samples_path = Path(result_dir) / f"{run_name}_samples.npy"
    probabilities_path = Path(result_dir) / f"{run_name}_probabilities.npy"
    row_cache_path = OOD_CACHE_DIR / f"{run_name}_ood_row.json"

    if not samples_path.exists():
        print(f"UEBERSPRUNGEN (nicht gefunden): {label} -> {samples_path}")
        continue

    if row_cache_path.exists():
        with open(row_cache_path, "r", encoding="utf-8") as f:
            row = json.load(f)
        results.append(row)
        print(f"\n=== {label} ({run_name}) === (aus Zwischenspeicherung geladen)")
        print(f"  ID  Mean MI: {row['id_mean_mi']:.4f}   OOD Mean MI: {row['ood_mean_mi']:.4f}")
        print(f"  AUROC (MI):      {row['auroc_mutual_information']:.4f}")
        print(f"  AUROC (Entropy): {row['auroc_predictive_entropy']:.4f}")
        continue

    print(f"\n=== {label} ({run_name}) ===")
    sample_thetas = np.load(samples_path)
    print("Samples shape:", sample_thetas.shape)

    # ---------- ID: bereits gespeicherte Wahrscheinlichkeiten wiederverwenden ----------
    if probabilities_path.exists():
        id_probs = np.load(probabilities_path)
        id_mean_probs = id_probs.mean(axis=0)
        print(f"ID-Wahrscheinlichkeiten aus dem Fitting-Notebook geladen: {probabilities_path.name}")
    else:
        print("Keine gespeicherten ID-Wahrscheinlichkeiten gefunden -- rechne neu.")
        eval_master_key, id_key = random.split(eval_master_key)
        id_probs, id_mean_probs, id_key = qpu.compute_posterior_predictive_probabilities(
            env=env, rebuild_full_params=rebuild_full_params,
            sample_thetas=sample_thetas, x_eval=x_id, y_eval=y_id,
            rng_key=id_key, eval_batch_size=EVAL_BATCH_SIZE,
        )

    id_metrics = qpu.compute_predictive_metrics(
        sample_probabilities=id_probs, mean_probabilities=id_mean_probs, y_true=y_id,
    )

    # ---------- OOD: HuffPost (Near-OOD) ----------
    eval_master_key, ood_key = random.split(eval_master_key)
    ood_probs, ood_mean_probs, ood_key = qpu.compute_posterior_predictive_probabilities(
        env=env, rebuild_full_params=rebuild_full_params,
        sample_thetas=sample_thetas, x_eval=x_ood, y_eval=y_ood,
        rng_key=ood_key, eval_batch_size=EVAL_BATCH_SIZE,
    )
    ood_metrics = qpu.compute_predictive_metrics(
        sample_probabilities=ood_probs, mean_probabilities=ood_mean_probs, y_true=y_ood,
    )
    # NUR die label-unabhaengigen Groessen sind fuer OOD gueltig:
    # predictive_entropy, expected_entropy, mutual_information.

    auroc_mi = compute_auroc(
        np.asarray(id_metrics["mutual_information"]),
        np.asarray(ood_metrics["mutual_information"]),
    )
    auroc_entropy = compute_auroc(
        np.asarray(id_metrics["predictive_entropy"]),
        np.asarray(ood_metrics["predictive_entropy"]),
    )

    row = {
        "label": label,
        "run_name": run_name,
        "n_samples": int(sample_thetas.shape[0]),
        "id_accuracy": float(id_metrics["accuracy"]),
        "id_nll": float(id_metrics["posterior_predictive_nll"]),
        "id_mean_mi": float(id_metrics["mutual_information"].mean()),
        "id_mean_entropy": float(id_metrics["predictive_entropy"].mean()),
        "ood_mean_mi": float(ood_metrics["mutual_information"].mean()),
        "ood_mean_entropy": float(ood_metrics["predictive_entropy"].mean()),
        "auroc_mutual_information": auroc_mi,
        "auroc_predictive_entropy": auroc_entropy,
    }
    results.append(row)

    with open(row_cache_path, "w", encoding="utf-8") as f:
        json.dump(row, f, indent=2)

    print(f"  ID  Mean MI: {row['id_mean_mi']:.4f}   OOD Mean MI: {row['ood_mean_mi']:.4f}")
    print(f"  AUROC (MI):      {auroc_mi:.4f}")
    print(f"  AUROC (Entropy): {auroc_entropy:.4f}")
    print(f"  Zwischengespeichert: {row_cache_path}")



=== MILE (Multi-Chain, K=3, N=2048, repariert) (mile_multichain_FIXED_chain1paramswap_donor23mean_balanced8192_seq32_prior0.02_warmup250_spc30_thin30_testeval) === (aus Zwischenspeicherung geladen)
  ID  Mean MI: 0.0079   OOD Mean MI: 0.0165
  AUROC (MI):      0.8137
  AUROC (Entropy): 0.9906

=== MFVI (N=2048) (mfvi_seed1_balanced8192_seq32_prior0.02_steps200_samples10_testeval) === (aus Zwischenspeicherung geladen)
  ID  Mean MI: 0.0219   OOD Mean MI: 0.0482
  AUROC (MI):      0.8881
  AUROC (Entropy): 0.9890

=== Laplace (N=2048) (laplace_seed1_balanced8192_seq32_prior0.02_fscale1.0_samples10) === (aus Zwischenspeicherung geladen)
  ID  Mean MI: 0.0568   OOD Mean MI: 0.0969
  AUROC (MI):      0.7672
  AUROC (Entropy): 0.9879

=== Deep Ensemble (N=3) (deep_ensemble_n3_seq32) === (aus Zwischenspeicherung geladen)
  ID  Mean MI: 0.0036   OOD Mean MI: 0.0145
  AUROC (MI):      0.8864
  AUROC (Entropy): 0.9927

=== pSMILE (Mini-Batch, K=3, N=2048) (mile_psmile_k3_prior0.02_mb32_spc90_th

Samples shape: (203, 540672)
ID-Wahrscheinlichkeiten aus dem Fitting-Notebook geladen: mile_psmile_k3_prior0.02_mb32_spc90_thin30_testeval_probabilities.npy


Accuracy: 0.8852631449699402
LPPD: -0.6340794563293457
Posterior predictive NLL: 0.6340794563293457
Brier Score: 0.3051702380180359
Mean predictive entropy: 1.087570071220398
Mean expected entropy: 0.749213695526123
Mean mutual information: 0.33835646510124207


Evaluation examples: 2000
Evaluation batch size: 64
Posterior samples: 203
Evaluating posterior sample 1/203


Evaluating posterior sample 2/203


...

Evaluating posterior sample 201/203


Evaluating posterior sample 202/203


Evaluating posterior sample 203/203



Evaluation completed
Probability tensor shape: (203, 2000, 4)
Expected shape: (203, 2000, 4)
All probabilities finite: True


Accuracy: 0.0005000000237487257
LPPD: -2.244537353515625
Posterior predictive NLL: 2.244537353515625
Brier Score: 1.0624276399612427
Mean predictive entropy: 1.3041248321533203
Mean expected entropy: 1.0203940868377686
Mean mutual information: 0.2837308347225189
  ID  Mean MI: 0.3384   OOD Mean MI: 0.2837
  AUROC (MI):      0.2107
  AUROC (Entropy): 0.9295
  Zwischengespeichert: /dss/dsshome1/00/ra58vit2/Masterarbeit/method_results/ood_cache_near_huffpost/mile_psmile_k3_prior0.02_mb32_spc90_thin30_testeval_ood_row.json


In [11]:
# ============================================================
# COMPARISON TABLE
# ============================================================

columns = [
    ("label", "Methode", 22),
    ("id_accuracy", "ID Acc.", 8),
    ("id_mean_mi", "ID MI", 8),
    ("ood_mean_mi", "OOD MI", 8),
    ("auroc_mutual_information", "AUROC(MI)", 10),
    ("auroc_predictive_entropy", "AUROC(Ent.)", 11),
]

header = "  ".join(name.rjust(width) for _, name, width in columns)
print(header)
print("-" * len(header))

for row in results:
    line_parts = []
    for key, _, width in columns:
        value = row[key]
        text = f"{value:.4f}" if isinstance(value, float) else str(value)
        line_parts.append(text.rjust(width))
    print("  ".join(line_parts))

print()
print("AUROC-Interpretation: 0.5 = MI/Entropie unterscheidet ID/OOD nicht")
print("besser als Zufall; naeher an 1.0 = MI/Entropie erkennt OOD gut.")
print()
print("Near-OOD (HuffPost) vs. Far-OOD (IMDB, siehe 07_ood_evaluation.ipynb):")
print("niedrigere AUROC-Werte hier sind zu erwarten und kein Fehler --")
print("Near-OOD ist per Konstruktion die schwerere Unterscheidungsaufgabe.")


               Methode   ID Acc.     ID MI    OOD MI   AUROC(MI)  AUROC(Ent.)
-----------------------------------------------------------------------------
MILE (Multi-Chain, K=3, N=2048, repariert)    0.9138    0.0079    0.0165      0.8137       0.9906
         MFVI (N=2048)    0.8945    0.0219    0.0482      0.8881       0.9890
      Laplace (N=2048)    0.9070    0.0568    0.0969      0.7672       0.9879
   Deep Ensemble (N=3)    0.9150    0.0036    0.0145      0.8864       0.9927
pSMILE (Mini-Batch, K=3, N=2048)    0.8853    0.3384    0.2837      0.2107       0.9295

AUROC-Interpretation: 0.5 = MI/Entropie unterscheidet ID/OOD nicht
besser als Zufall; naeher an 1.0 = MI/Entropie erkennt OOD gut.

Near-OOD (HuffPost) vs. Far-OOD (IMDB, siehe 07_ood_evaluation.ipynb):
niedrigere AUROC-Werte hier sind zu erwarten und kein Fehler --
Near-OOD ist per Konstruktion die schwerere Unterscheidungsaufgabe.


In [12]:
# ============================================================
# SAVE CONSOLIDATED OOD COMPARISON
# ============================================================

with open(OOD_RESULT_PATH, "w", encoding="utf-8") as f:
    json.dump({
        "ood_dataset": "huffpost_news_category (near-ood: TRAVEL, STYLE & BEAUTY, PARENTING, FOOD & DRINK)",
        "n_id_examples": int(y_id.shape[0]),
        "n_ood_examples": int(y_ood.shape[0]),
        "seq_len": SEQ_LEN,
        "categories": HUFFPOST_CATEGORIES,
        "n_per_category": N_PER_CATEGORY,
        "results": results,
    }, f, indent=2)

print("Gespeichert:", OOD_RESULT_PATH)


Gespeichert: /dss/dsshome1/00/ra58vit2/Masterarbeit/method_results/ood_huffpost_near_comparison.json
